# Module 35 — MapReduce Simulation with Pandas

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, NumPy, Plotly  
> **Prerequisite**: Module 2 (Advanced GroupBy)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Web Server Logs](#3-synthetic-web-server-logs)
4. [Map Phase](#4-map-phase)
5. [Shuffle & Sort Phase](#5-shuffle--sort-phase)
6. [Reduce Phase](#6-reduce-phase)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why MapReduce for big data?

MapReduce is **fundamental** for distributed data processing:
- **Scalability**: Process petabytes of data across clusters.
- **Fault tolerance**: Automatically handles node failures.
- **Simplicity**: Map and Reduce are intuitive operations.

**Applications at Yandex**:
1. **Web log analysis**: Process billions of daily page views.
2. **Search indexing**: Build inverted indexes for search.
3. **Click analytics**: Aggregate click-through rates.

**Key concepts**:
- **Map**: Transform input into key-value pairs.
- **Shuffle**: Group values by key.
- **Reduce**: Aggregate values for each key.

In this notebook we will:
1. Generate synthetic web server logs (10M rows).
2. Simulate Map, Shuffle, and Reduce phases.
3. Implement word count and aggregation patterns.
4. Compare with native Pandas operations.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Web Server Logs

In [ ]:
# Generate synthetic web server logs
N_ROWS = 10_000_000

# Generate data
data = {
    'timestamp': pd.date_range('2024-01-01', periods=N_ROWS, freq='1s'),
    'user_id': np.random.randint(0, 1000000, N_ROWS),
    'page': np.random.choice(['home', 'search', 'product', 'cart', 'checkout'], N_ROWS, p=[0.3, 0.3, 0.2, 0.1, 0.1]),
    'country': np.random.choice(['US', 'UK', 'DE', 'FR', 'RU', 'JP'], N_ROWS, p=[0.3, 0.2, 0.15, 0.1, 0.15, 0.1]),
    'duration': np.random.exponential(30, N_ROWS),
    'status_code': np.random.choice([200, 301, 404, 500], N_ROWS, p=[0.85, 0.05, 0.08, 0.02]),
}

df = pd.DataFrame(data)

print(f"✓ Generated {N_ROWS:,} web server log entries")
print(f"  Users: {df['user_id'].nunique():,}")
print(f"  Pages: {df['page'].nunique()}")
print(f"  Countries: {df['country'].nunique()}")
print(f"\nSample logs:")
print(df.head())

---
## §4 · Map Phase

In [ ]:
# Map phase: Transform data into key-value pairs
def map_phase(df, key_col, value_col):
    """Map phase: emit (key, value) pairs."""
    return df[[key_col, value_col]].values.tolist()

# Example: Count page views by page
print("Map Phase: Emit (page, 1) pairs")
t0 = time.time()
mapped = map_phase(df, 'page', 'status_code')
map_time = time.time() - t0

print(f"  Mapped {len(mapped):,} pairs in {map_time:.2f}s")
print(f"  Sample: {mapped[:3]}")

---
## §5 · Shuffle & Sort Phase

In [ ]:
# Shuffle phase: Group values by key
def shuffle_phase(mapped_pairs):
    """Shuffle phase: group values by key."""
    shuffled = {}
    for key, value in mapped_pairs:
        if key not in shuffled:
            shuffled[key] = []
        shuffled[key].append(value)
    return shuffled

# Shuffle
print("Shuffle Phase: Group by key")
t0 = time.time()
shuffled = shuffle_phase(mapped)
shuffle_time = time.time() - t0

print(f"  Shuffled into {len(shuffled)} groups in {shuffle_time:.2f}s")
print(f"  Groups: {list(shuffled.keys())}")

---
## §6 · Reduce Phase

In [ ]:
# Reduce phase: Aggregate values for each key
def reduce_phase(shuffled, reduce_fn):
    """Reduce phase: aggregate values for each key."""
    results = {}
    for key, values in shuffled.items():
        results[key] = reduce_fn(values)
    return results

# Reduce: Count page views
print("Reduce Phase: Count page views")
t0 = time.time()
reduced = reduce_phase(shuffled, len)
reduce_time = time.time() - t0

print(f"  Reduced in {reduce_time:.2f}s")
print(f"\nPage view counts:")
for page, count in sorted(reduced.items(), key=lambda x: -x[1]):
    print(f"  {page}: {count:,}")

# Compare with native Pandas
print("\n" + "=" * 70)
print("Comparison with native Pandas:")
t0 = time.time()
pandas_result = df.groupby('page').size()
pandas_time = time.time() - t0

print(f"  Pandas groupby: {pandas_time:.4f}s")
print(f"  MapReduce simulation: {map_time + shuffle_time + reduce_time:.4f}s")
print(f"\n💡 Pandas is optimized for in-memory operations")

---
## §7 · Visualization

In [ ]:
# Visualize MapReduce results
fig = make_subplots(rows=1, cols=2, subplot_titles=['Page Views by Page', 'Status Code Distribution'])

# Page views
pages = list(reduced.keys())
counts = list(reduced.values())

fig.add_trace(go.Bar(
    x=pages,
    y=counts,
    name='Page Views',
    marker_color='#636EFA'
), row=1, col=1)

# Status codes
status_counts = df['status_code'].value_counts().sort_index()
fig.add_trace(go.Bar(
    x=status_counts.index.astype(str),
    y=status_counts.values,
    name='Status Codes',
    marker_color='#EF553B'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - MapReduce pattern is intuitive for aggregation tasks")
print("   - Pandas is much faster for in-memory operations")
print("   - MapReduce shines when data exceeds memory")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Data Engineering Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Large-scale aggregation, log processing, ETL |
> | **Framework** | Hadoop, Spark, or cloud-native (BigQuery, Redshift) |
> | **Pattern** | Map → Shuffle → Reduce for aggregations |
> | **Optimization** | Combiner for local aggregation, partitioner for balance |
> | **Scale** | Petabytes of data across distributed clusters |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Log processing pipeline**:
>    ```
>    Logs → Map (parse) → Shuffle (group) → Reduce (aggregate) → Store
>    ```
>
> 2. **Common patterns**:
>    | Pattern | Map | Reduce |
>    |---------|-----|--------|
>    | Word count | (word, 1) | Sum |
>    | Page views | (page, 1) | Count |
>    | Avg duration | (page, duration) | Mean |
>
> 3. **When to use MapReduce vs Pandas**:
>    - **Pandas**: Data fits in memory (< 100GB)
>    - **MapReduce**: Data exceeds memory, needs distribution
>    - **Spark**: Best of both (in-memory + distributed)
>
> ### 🔑 Key Takeaways
>
> 1. **MapReduce is foundational** for distributed data processing.
> 2. **The pattern is simple**: Map → Shuffle → Reduce.
> 3. **Pandas is faster** for in-memory operations.
> 4. **Use distributed frameworks** when data exceeds memory.
> 5. **Combiners optimize** network transfer in MapReduce.